# Comparar soluciones para mi proyecto (M5, tienda CA_1)

Predecir las unidades vendidas de un item en la tienda al dia siguiente. La preparacion y la division por fecha (ultimas 8 semanas como prueba) se importan de `pipeline_proyecto.py`. Para entrenar rapido se usa una muestra de 200 000 filas de entrenamiento; la prueba (170 744 filas) va completa. Metrica: error absoluto medio (MAE), en unidades.

In [1]:
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

from pipeline_proyecto import cargar_y_dividir, construir_preparacion

X_ent, X_prueba, y_ent, y_prueba = cargar_y_dividir(maximo_entrenamiento=200_000)
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)

Entrenamiento: (200000, 9) Prueba: (170744, 9)


## Referencias sin modelo

In [2]:
referencias = {
    "siempre la media": np.full(len(y_prueba), y_ent.mean()),
    "ventas de hace 7 dias": X_prueba["units_lag_7"].values,
    "media de los 28 dias previos": X_prueba["units_media_28"].values,
}
for nombre, pred in referencias.items():
    print(f"{nombre:30s} MAE prueba {mean_absolute_error(y_prueba, pred):.4f}")

siempre la media               MAE prueba 1.8613
ventas de hace 7 dias          MAE prueba 1.3364
media de los 28 dias previos   MAE prueba 1.1082


## Los dos modelos

In [3]:
resultados = {}
entrenados = {}


def evaluar(nombre, modelo):
    pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", modelo)])
    inicio = time.perf_counter()
    pipeline.fit(X_ent, y_ent)
    segundos = time.perf_counter() - inicio
    mae_ent = mean_absolute_error(y_ent, pipeline.predict(X_ent))
    mae_prueba = mean_absolute_error(y_prueba, pipeline.predict(X_prueba))
    resultados[nombre] = {"modelo": nombre, "mae_entrenamiento": round(mae_ent, 4),
                          "mae_prueba": round(mae_prueba, 4), "segundos": round(segundos, 2)}
    entrenados[nombre] = pipeline
    print(f"{nombre:20s} entrenamiento {mae_ent:.4f}   prueba {mae_prueba:.4f}   {segundos:7.2f} s")

In [4]:
evaluar("gradient boosting", GradientBoostingRegressor(random_state=42))
ensamblaje = entrenados["gradient boosting"]

gradient boosting    entrenamiento 1.1470   prueba 1.0895     15.83 s


In [5]:
evaluar("red neuronal", MLPRegressor(hidden_layer_sizes=(64,), max_iter=1000, random_state=42))
red = entrenados["red neuronal"]
modelo_red = red.named_steps["modelo"]
parametros_red = sum(m.size for m in modelo_red.coefs_) + sum(v.size for v in modelo_red.intercepts_)
print("Parametros de la red:", parametros_red, "| iteraciones:", modelo_red.n_iter_)

red neuronal         entrenamiento 1.1307   prueba 1.0713     73.29 s
Parametros de la red: 2177 | iteraciones: 83


In [6]:
pd.DataFrame(resultados.values())

,modelo,mae_entrenamiento,mae_prueba,segundos
0,gradient boosting,1.1470,1.0895,15.83
1,red neuronal,1.1307,1.0713,73.29


## La prueba del LLM: 10 filas de prueba

In [7]:
LEGIBLES = ["dept_id", "wday", "month", "sell_price", "is_snap", "is_event_day", "units_lag_7", "units_media_28"]
rng_ej = X_ent.sample(30, random_state=7)
y_ej = y_ent.loc[rng_ej.index]
muestra = X_prueba.sample(10, random_state=7)
y_muestra = y_prueba.loc[muestra.index]

ejemplos_csv = rng_ej[LEGIBLES].copy()
ejemplos_csv["units"] = y_ej.values
texto = (
    "Estimas las unidades vendidas de un producto de Walmart en una tienda de California en un dia. "
    "Abajo hay 30 casos con las unidades reales y 10 casos sin ellas. Responde solo con 10 numeros, "
    "uno por linea, en el mismo orden de los casos sin unidades, sin texto adicional. "
    "units_lag_7 son las unidades de hace 7 dias y units_media_28 el promedio de los 28 dias previos; "
    "wday va de 1 (sabado) a 7 (viernes).\n\n"
    "Casos con unidades:\n" + ejemplos_csv.to_csv(index=False)
    + "\nCasos sin unidades:\n" + muestra[LEGIBLES].to_csv(index=False)
)
with open("prompt_llm_proyecto.txt", "w") as archivo:
    archivo.write(texto)
print("prompt_llm_proyecto.txt escrito:", len(texto), "caracteres")

prompt_llm_proyecto.txt escrito: 2402 caracteres


In [8]:
print("MAE en las 10 filas")
print("  gradient boosting:", round(mean_absolute_error(y_muestra, ensamblaje.predict(muestra)), 4))
print("  red neuronal     :", round(mean_absolute_error(y_muestra, red.predict(muestra)), 4))
print("  media 28 dias    :", round(mean_absolute_error(y_muestra, muestra["units_media_28"]), 4))

MAE en las 10 filas
  gradient boosting: 1.4052
  red neuronal     : 1.0183
  media 28 dias    : 1.3964


In [9]:
with open("respuestas_llm_proyecto.txt") as archivo:
    predicciones_llm = [float(linea) for linea in archivo if linea.strip()]
assert len(predicciones_llm) == 10
mae_llm = mean_absolute_error(y_muestra, predicciones_llm)
print(f"MAE del LLM en las 10 filas: {mae_llm:.4f}")
print(pd.DataFrame({"real": y_muestra.values, "llm": predicciones_llm,
                    "gradient boosting": ensamblaje.predict(muestra).round(2),
                    "red neuronal": red.predict(muestra).round(2),
                    "media 28 dias": muestra["units_media_28"].round(2).values}))

MAE del LLM en las 10 filas: 1.6000
   real  llm  gradient boosting  red neuronal  media 28 dias
0     0  0.0               0.36          0.23           0.39
1     1  0.0               0.61          0.58           0.71
2    16  7.0               8.31         11.36           8.79
3     3  0.0               0.83          0.46           0.93
4     2  3.0               3.39          2.18           4.14
5     4  4.0               3.61          3.38           3.46
6     1  0.0               0.47          0.48           0.46
7     1  1.0               1.33          1.29           1.07
8     0  0.0               0.21          0.14           0.18
9     1  0.0               0.41          0.39           0.46


In [10]:
# Costo mensual del LLM: GPT-6.1 Terra, 1.00 USD por millon de tokens de entrada.
# Un prompt de 10 casos, 1 token cada 4 caracteres. Se estima un item en la
# tienda CA_1 cada dia: unos 3 000 items x 30 dias.
PRECIO_POR_MILLON_DE_TOKENS = 1.00
tokens_por_prompt = len(texto) / 4
predicciones_al_mes = 3_000 * 30
costo_mes = tokens_por_prompt * (predicciones_al_mes / 10) * PRECIO_POR_MILLON_DE_TOKENS / 1_000_000
print(f"{tokens_por_prompt:.0f} tokens por prompt de 10 casos -> USD {costo_mes:.2f} al mes para {predicciones_al_mes} predicciones")

600 tokens por prompt de 10 casos -> USD 5.40 al mes para 90000 predicciones
